# Explore case events through a DHIS2-shaped adapter

This is a **local course simulation**, not a DHIS2 server. It uses synthetic measles case events. The adapter answers three read-only requests in the Python Jupyter kernel; it makes no network requests or changes to a real health information system.

*Unit 3: Using Informatics to Get Information Faster.* Learning objective: **Use software to explore a dataset.**

The [UCSF course book's Unit 2.1 Assignment 2](https://github.com/UCSF-IGHS/FETP/blob/main/FETP_book/Unit_2.1_Case_study_Assignment_2.ipynb) asks: “How big is this dataset?”, “What timeframe does the dataset span?”, and “What locations does the dataset include?” [Assignment 3](https://github.com/UCSF-IGHS/FETP/blob/main/FETP_book/Unit_2.1_Case_study_Assignment_3.ipynb) adds: “Describe the variables in your dataset.” Those are the questions this exercise preserves.

In the course assignment, the network is down and the district data manager supplies an Excel backup and DHIS2 data dictionary. This pilot provides a synthetic, local API-shaped copy of case events and dictionary entries so you can practice the same inspection questions inside JupyterLite. It does not simulate an online DHIS2 connection.

## Your task

1. Use `api.get` to find the ID of **Kambia District** in `/api/organisationUnits`.
2. Query `/api/tracker/events` with `program="P_MEASLES"` and that `orgUnit` ID. Store the returned list in `events`.
3. Use `/api/dataElements` to find the ID for **Date of rash onset**. Store it in `onset_element`.
4. From the returned events, compute `n_cases`, `n_missing_onset`, `first_onset`, and `last_onset`. For the date range, use nonblank onset dates, as `YYYY-MM-DD` strings.

Run the setup cell, then edit and run the exercise cell. The **Learning** panel checks your result. The one missing onset date is a data-quality signal; do not silently replace it with the event date.


In [ ]:
import json
FIXTURE = json.loads('{"organisationUnits":[{"id":"OU_KAMBIA","displayName":"Kambia District"},{"id":"OU_PORT_LOKO","displayName":"Port Loko District"}],"dataElements":[{"id":"DE_ONSET","displayName":"Date of rash onset","valueType":"DATE"},{"id":"DE_AGE","displayName":"Age in years","valueType":"INTEGER"},{"id":"DE_CLASS","displayName":"Case classification","valueType":"TEXT"}],"events":[{"event":"EV_001","program":"P_MEASLES","orgUnit":"OU_KAMBIA","occurredAt":"2021-10-20","dataValues":[{"dataElement":"DE_ONSET","value":"2021-10-18"},{"dataElement":"DE_AGE","value":"4"},{"dataElement":"DE_CLASS","value":"suspected"}]},{"event":"EV_002","program":"P_MEASLES","orgUnit":"OU_KAMBIA","occurredAt":"2021-10-23","dataValues":[{"dataElement":"DE_ONSET","value":"2021-10-21"},{"dataElement":"DE_AGE","value":"8"},{"dataElement":"DE_CLASS","value":"suspected"}]},{"event":"EV_003","program":"P_MEASLES","orgUnit":"OU_KAMBIA","occurredAt":"2021-10-27","dataValues":[{"dataElement":"DE_ONSET","value":""},{"dataElement":"DE_AGE","value":"2"},{"dataElement":"DE_CLASS","value":"suspected"}]},{"event":"EV_004","program":"P_MEASLES","orgUnit":"OU_KAMBIA","occurredAt":"2021-11-02","dataValues":[{"dataElement":"DE_ONSET","value":"2021-10-30"},{"dataElement":"DE_AGE","value":"13"},{"dataElement":"DE_CLASS","value":"probable"}]},{"event":"EV_005","program":"P_MEASLES","orgUnit":"OU_KAMBIA","occurredAt":"2021-11-05","dataValues":[{"dataElement":"DE_ONSET","value":"2021-11-04"},{"dataElement":"DE_AGE","value":"6"},{"dataElement":"DE_CLASS","value":"confirmed"}]},{"event":"EV_006","program":"P_MEASLES","orgUnit":"OU_PORT_LOKO","occurredAt":"2021-10-25","dataValues":[{"dataElement":"DE_ONSET","value":"2021-10-24"},{"dataElement":"DE_AGE","value":"5"},{"dataElement":"DE_CLASS","value":"suspected"}]},{"event":"EV_007","program":"P_MEASLES","orgUnit":"OU_PORT_LOKO","occurredAt":"2021-11-03","dataValues":[{"dataElement":"DE_ONSET","value":"2021-11-01"},{"dataElement":"DE_AGE","value":"7"},{"dataElement":"DE_CLASS","value":"probable"}]}]}')
from copy import deepcopy

class CourseDhis2Adapter:
    """A read-only, in-memory subset of DHIS2-like GET responses."""

    def __init__(self, fixture):
        self._fixture = deepcopy(fixture)

    def get(self, path, params=None):
        params = dict(params or {})
        if path == "/api/organisationUnits":
            if params:
                raise ValueError("organisationUnits does not support query parameters in this pilot")
            return {"organisationUnits": deepcopy(self._fixture["organisationUnits"])}
        if path == "/api/dataElements":
            if params:
                raise ValueError("dataElements does not support query parameters in this pilot")
            return {"dataElements": deepcopy(self._fixture["dataElements"])}
        if path == "/api/tracker/events":
            if set(params) != {"program", "orgUnit"}:
                raise ValueError("events requires program and orgUnit")
            if params["program"] != "P_MEASLES":
                raise ValueError("unknown program")
            if params["orgUnit"] not in {u["id"] for u in self._fixture["organisationUnits"]}:
                raise ValueError("unknown organisation unit")
            return {"events": deepcopy([e for e in self._fixture["events"]
                                        if e["program"] == params["program"]
                                        and e["orgUnit"] == params["orgUnit"]])}
        raise ValueError("unsupported endpoint: " + str(path))

api = CourseDhis2Adapter(FIXTURE)


In [ ]:
# Use the adapter and its data dictionary. Replace each None.
org_unit = None
events = None
onset_element = None
n_cases = None
n_missing_onset = None
first_onset = None
last_onset = None


## References

- [UCSF FETP JupyterBook: Unit 2.1 case study, Assignment 2](https://github.com/UCSF-IGHS/FETP/blob/main/FETP_book/Unit_2.1_Case_study_Assignment_2.ipynb): dataset size, timeframe, locations, and population.
- [UCSF FETP JupyterBook: Unit 2.1 case study, Assignment 3](https://github.com/UCSF-IGHS/FETP/blob/main/FETP_book/Unit_2.1_Case_study_Assignment_3.ipynb): use the data dictionary to understand variables.
- [FETP Unit 3 instructor guide](https://docs.google.com/document/d/1B6qrW7QBcL4uCO85QPIN8pJ--37S8N6j3xkT87qBACA).
- [FETP Unit 3 slides](https://docs.google.com/presentation/d/1EXkFCxffubjhpM94373WiJ0a9_V0Tsfn): dataset exploration, slides 56–57.
- [FETP Unit 3 knowledge check](https://forms.gle/Y4yG5gMbFznczrBd9).
- [DHIS2 Tracker API reference](https://docs.dhis2.org/en/develop/using-the-api/dhis-core-version-241/tracker.html): event query shape. This pilot implements only the documented subset stated above.

*The case events and IDs in this notebook are synthetic.*
